<a href="https://colab.research.google.com/github/EAwoyemi110/Ai-job-market-dataset-Analysis/blob/main/Copy_of_Text_Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install "datasets==3.6.0"

In [ ]:
!pip install hf_transfer

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "12000" # Increased timeout to 600 seconds
import json
import random
import warnings

import torch
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from scipy.stats import pointbiserialr, mannwhitneyu
!pip install statsmodels
from statsmodels.stats.multitest import multipletests
from transformers import (
    AutoModel,
    AutoImageProcessor,
    AutoProcessor,
    PaliGemmaForConditionalGeneration,
)
from huggingface_hub import login
login()

In [ ]:
SIGLIP_CKPT = "google/siglip-so400m-patch14-384"
PALIGEMMA_CKPT = "google/paligemma2-3b-pt-224"

CANVAS_SIZE = 384 #fixed square size that vision processor resizes every image to (is this right size?)

RESULTS_DIR = "results/exp2"

VG_HF_NAME = "ranjaykrishna/visual_genome"
VG_HF_CONFIG = "relationships_v1.2.0"

SPATIAL_PREDICATES = [
    "on", "on top of", "under", "below", "above", "in front of", "behind",
    "next to", "near", "inside", "left of", "right of",
]

In [ ]:
N_IMAGES_PER_DATASET = 20 # Reduced the number of images significantly to try and avoid timeout
#Change back to 20 later
FDR_ALPHA = 0.05

SEED = 0
random.seed(SEED)

model_vit = AutoModel.from_pretrained(SIGLIP_CKPT)
model_vlm = PaliGemmaForConditionalGeneration.from_pretrained(PALIGEMMA_CKPT)
vision_tower = model_vlm.model.vision_tower

model_vit.eval()
model_vlm.eval()

vision_processor = AutoImageProcessor.from_pretrained(SIGLIP_CKPT)
vlm_processor = AutoProcessor.from_pretrained(PALIGEMMA_CKPT)

PATCH_SIZE_VIT = model_vit.config.vision_config.patch_size
PATCH_SIZE_VLM = model_vlm.config.vision_config.patch_size
if PATCH_SIZE_VIT != PATCH_SIZE_VLM:
  warnings.warn(
      f"patch size mismatch: SigLIP={PATCH_SIZE_VIT}, PaliGemma vision "
      f"tower={PATCH_SIZE_VLM}. Exp 1's shared-preprocessing approach "
      f"assumes these match; re-derive patch indices per-tower if not."
  )

GRID_SIZE = CANVAS_SIZE // PATCH_SIZE_VIT


activations_vit = {}
activations_vlm = {}

In [ ]:
import datasets
def get_hook(store, name):
  def hook(module, input, output):
    store[name] = output.detach().cpu()
  return hook


def is_block(module):
  return type(module).__name__.endswith("Layer")


for name, module in model_vit.named_modules():
  if is_block(module):
    module.register_forward_hook(get_hook(activations_vit, name))

for name, module in vision_tower.named_modules():
  if is_block(module):
    module.register_forward_hook(get_hook(activations_vlm, name))

    #RETYPE LATER
BASE_OBJECT_FEATURES = {
    "object_id": datasets.Value("int32"),
    "x": datasets.Value("int32"),
    "y": datasets.Value("int32"),
    "w": datasets.Value("int32"),
    "h": datasets.Value("int32"),
    "names": [datasets.Value("string")],
    "synsets": [datasets.Value("string")],
}

RELATIONSHIP_FEATURES = {
    "relationship_id": datasets.Value("int32"),
    "predicate": datasets.Value("string"),
    "synsets": [datasets.Value("string")],  # fixed: script wrongly declares this as a single string
    "subject": BASE_OBJECT_FEATURES,
    "object": BASE_OBJECT_FEATURES,
}

VG_FEATURES = datasets.Features({
    "image": datasets.Image(),
    "image_id": datasets.Value("int32"),
    "url": datasets.Value("string"),
    "width": datasets.Value("int32"),
    "height": datasets.Value("int32"),
    "coco_id": datasets.Value("int64"),
    "flickr_id": datasets.Value("int64"),
    "relationships": [RELATIONSHIP_FEATURES],
})
    #RETYPE LATER


def load_vg(hf_name=VG_HF_NAME, hf_config=VG_HF_CONFIG, n_images=None):
  from datasets import load_dataset
  import itertools

  vg = load_dataset(hf_name, hf_config, split="train", trust_remote_code=True, streaming=True, features=VG_FEATURES)
  if n_images is not None:
    vg = itertools.islice(vg, n_images)

  records = []
  for entry in vg:
    image = entry["image"]

    for rel in entry["relationships"]:
      predicate = rel["predicate"].strip().lower()
      if predicate not in SPATIAL_PREDICATES:
        continue

      subj = rel["subject"]
      obj = rel["object"]
      subj_name = subj.get("name") or (subj.get("names") or [None])[0]
      obj_name = obj.get("name") or (obj.get("names") or [None])[0]
      if subj_name is None or obj_name is None:
       continue

      subj_bbox = [subj["x"], subj["y"], subj["x"] + subj["w"], subj["y"] + subj["h"]]
      obj_bbox = [obj["x"], obj["y"], obj["x"] + obj["w"], obj["y"] + obj["h"]]

      records.append({
          "dataset": "vg",
          "image": image,
          "obj_a_bbox": subj_bbox,
          "obj_b_bbox": obj_bbox,
          "question": f"True or False: the {subj_name} is {predicate} the {obj_name}",
          "answer": "true",
      })

      wrong_predicate = random.choice([p for p in SPATIAL_PREDICATES if p!= predicate])
      records.append({
          "dataset": "vg",
          "image": image,
          "obj_a_bbox": subj_bbox,
          "obj_b_bbox": obj_bbox,
          "question": f"True or False: the {subj_name} is {wrong_predicate} the {obj_name}",
          "answer": "false",
      })

  return records


def rescale_bbox(bbox, orig_size, target_size=(CANVAS_SIZE, CANVAS_SIZE)):
  ow, oh = orig_size
  tw, th = target_size
  x1, y1, x2, y2 = bbox
  return [x1 * tw / ow, y1 * th / oh, x2 * tw / ow, y2 * th / oh]


def bbox_to_patch_indices(bbox, canvas_size=CANVAS_SIZE, patch_size=PATCH_SIZE_VIT):
  x1, y1, x2, y2 = bbox
  grid_size = canvas_size // patch_size
  px1, py1 = int(x1 // patch_size), int(y1 // patch_size)
  px2, py2 = int(x2 // patch_size), int(y2 // patch_size)

  indices = []
  for py in range(max(py1, 0), min(py2 + 1, grid_size)):
    for px in range(max(px1, 0), min(px2 + 1, grid_size)):
      indices.append(py * grid_size + px)
  return indices

def run_crowding_pass(img):
  activations_vit.clear()
  activations_vlm.clear()
  inputs = vision_processor(images=img, return_tensors="pt")
  with torch.no_grad():
    _ = model_vit(**inputs)
    _ = vision_tower(inputs["pixel_values"])


def compute_pair_similarity(layer_acts, patch_idx_a, patch_idx_b):
  vec_a = layer_acts[patch_idx_a].mean(dim=0)
  vec_b = layer_acts[patch_idx_b].mean(dim=0)
  a = F.normalize(vec_a, dim=0)
  b = F.normalize(vec_b, dim=0)
  return (a @ b).item()



def run_accuracy_pass(img, question, expected_answer):
  prompt = f"answer en {question}"
  inputs = vlm_processor(text=prompt, images=img, return_tensors="pt")
  with torch.no_grad():
    output_ids = model_vlm.generate(**inputs, max_new_tokens=10)
  decoded = vlm_processor.batch_decode(output_ids, skip_special_tokens=True)[0]
  predicted = decoded.strip().lower()
  correct = predicted.startswith(expected_answer.strip().lower())
  return predicted, correct


In [27]:
os.makedirs(RESULTS_DIR, exist_ok=True)

records = load_vg(n_images=N_IMAGES_PER_DATASET)

if N_IMAGES_PER_DATASET is not None:
  records = records[:N_IMAGES_PER_DATASET]

crowding_rows = []

for rec in records:
  img = rec["image"].convert("RGB")
  orig_size = img.size

  run_crowding_pass(img)

  bbox_a = rescale_bbox(rec["obj_a_bbox"], orig_size)
  bbox_b = rescale_bbox(rec["obj_b_bbox"], orig_size)
  patch_idx_a = bbox_to_patch_indices(bbox_a)
  patch_idx_b = bbox_to_patch_indices(bbox_b)

  if not patch_idx_a or not patch_idx_b:
    continue

  predicted, correct = run_accuracy_pass(img, rec["question"], rec["answer"])


  for condition_name, activations in [
      ("vit_only", activations_vit),
      ("vlm_no_prompt", activations_vlm),
  ]:
    for layer_name, layer_acts in activations.items():
      acts = layer_acts.squeeze(0)
      sim = compute_pair_similarity(acts, patch_idx_a, patch_idx_b)
      crowding_rows.append({
          "dataset": rec["dataset"],
          "image_id": id(rec["image"]),
          "condition": condition_name,
          "layer": layer_name,
          "target_distractor_similarity": sim,
          "predicted_answer": predicted,
          "correct": correct,
      })

with open(os.path.join(RESULTS_DIR, "raw_results.json"), "w") as f:
  json.dump(crowding_rows, f)

print(f"Saved {len(crowding_rows)} result rows to {RESULTS_DIR}/raw_results.json")

ValueError: You have to specify input_ids

In [ ]:
df = pd.DataFrame(crowding_rows)

stat_rows = []
for condition in df["condition"].unique():
  layer_names, pvals, rvals = [], [], []
  for layer in df["layer"].unique():
    sub = df[(df["condition"] == condition) & (df["layer"] == layer)]
    if sub["correct"].nunique() < 2:
      continue
    r, p = pointbiserialr(sub["correct"].astype(int), sub["target_distractor_similarity"])
    layer_names.append(layer)
    pvals.append(p)
    rvals.append(r)

  if not pvals:
    continue

  reject, p_adj, _, _ = multipletests(pvals, alpha=FDR_ALPHA, method="fdr_bh")

  for layer, r, p_raw, p_corrected, sig in zip(layer_names, rvals, pvals, p_adj, reject):
    stat_rows.append({
        "condition": condition,
        "layer": layer,
        "r": r,
        "p_raw": p_raw,
        "p_fdr": p_corrected,
        "significant_after_correction": bool(sig),
        "effect_size_r": abs(r),
    })

stat_df = pd.DataFrame(stat_rows)
stat_df.to_csv(os.path.join(RESULTS_DIR, "correlation_by_layer.csv"), index=False)
print(f"Saved per-layer correlation table to {RESULTS_DIR}/correlation_by_layer.csv")



In [ ]:
fallback_rows = []
for condition in df["condition"].unique():
  for layer in df["layer"].unique():
    sub = df[(df["condition"] == condition) & (df["layer"] == layer)]
    correct_sims = sub.loc[sub["correct"], "target_distractor_similarity"]
    incorrect_sims = sub.loc[~sub["correct"], "target_distractor_similarity"]
    if len(correct_sims) < 2 or len(incorrect_sims) < 2:
      continue

    stat, p = mannwhitneyu(incorrect_sims, correct_sims, alternative="greater")

    n1, n2 = len(incorrect_sims), len(correct_sims)
    effect_size_rank_biserial = 1 - (2 * stat) / (n1 * n2)

    fallback_rows.append({
        "condition": condition,
        "layer": layer,
        "n_incorrect": n1,
        "n_correct": n2,
        "p_raw": p,
        "effect_size_rank_biserial": effect_size_rank_biserial,
    })


fallback_df = pd.DataFrame(fallback_rows)
if not fallback_df.empty:
  reject, p_adj, _, _ = multipletests(fallback_df["p_raw"], alpha=FDR_ALPHA, method="fdr_bh")
  fallback_df["p_fdr"] = p_adj
  fallback_df["significant_after_correction"] = reject
fallback_df.to_csv(os.path.join(RESULTS_DIR, "error_level_fallback.csv"), index=False)
print(f"Saved fallback error-level analysis to {RESULTS_DIR}/error_level_fallback.csv")